# OASIS-2 full-cohort MRI quality audit (private, resumable, CPU)

This notebook audits existing mounted MRI pairs one at a time. It uses the **same
shared validator as notebook 02**, including verified `(X,Y,Z,1)` array views.
No images, headers, affines, clinical manifests, diagnoses or labels are written
or changed. Only aggregate checkpoint/report JSON is written to a **dedicated
private output directory**, outside the raw and metadata directories.

The owner reports 150 subjects, 373 clinical sessions, 1,368 image-pair records,
3/3 sampled pairs passing, NIfTI-1 pair headers and `(256,256,128,1)` shapes, safe
subject-level split planning and prior preservation checks. These are supplied
reference observations; **Codex has not accessed the private dataset**. No final
clinical diagnoses or Task A/C labels are approved. A full technical pass does
not approve anatomical orientation, native-T1 provenance, visual QC or labels.

Keep this working notebook and all outputs private. Colab can save outputs and
edited paths into the notebook. Clear outputs and restore placeholders before
sharing. Never display source rows, filenames, IDs, image slices or raw errors.

## 1. Load the reviewed code-only bundle

These changes are unpublished. Do not clone an older branch and assume it has
the new audit modules. Obtain `oasis2-full-quality-audit-code.zip` from the current
reviewed workspace, built with `scripts/build_oasis2_audit_bundle.py`. Upload only
that **public code** archive using the cell below. It contains no datasets,
credentials, checkpoints or private reports. Public packages can be installed
from PyPI in Colab; no medical data is downloaded. Use Python 3.12+ and a CPU
runtime. A checksummed archive is not authenticated: use your reviewed bundle.

After a runtime restart, upload the same code bundle again. To change code,
restart with a new code directory and a new private audit output directory;
old checkpoints intentionally refuse changed code/parameters. Notebook 02 also
uses this extracted code directory after its shared-validator refactor.

In [ ]:
import sys
from pathlib import Path

CODE_ROOT = Path("/content/oasis2-audit-code")
BUNDLE_NAME = "oasis2-full-quality-audit-code.zip"
PUBLIC_PACKAGES = ("numpy==2.5.4", "nibabel==5.4.2")
INSTALL_PUBLIC_DEPENDENCIES = True

In [ ]:
import hashlib
import io
import json
import zipfile


def unpack_code_bundle(payload, destination):
    """Validate the entire code archive before writing to an ephemeral code folder."""
    destination = Path(destination)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        infos = archive.infolist()
        names = [info.filename for info in infos]
        if len(names) != len(set(names)) or len(names) > 100:
            raise ValueError("invalid_code_archive")
        if sum(info.file_size for info in infos) > 10 * 1024 * 1024:
            raise ValueError("code_archive_too_large")
        for info in infos:
            name = Path(info.filename)
            if (name.is_absolute() or ".." in name.parts or "\\" in info.filename
                    or info.is_dir() or info.file_size > 2 * 1024 * 1024
                    or ((info.external_attr >> 16) & 0o170000) == 0o120000):
                raise ValueError("unsafe_code_archive_member")
            allowed = (info.filename.startswith("src/brainagenticthon/") and name.suffix == ".py"
                       or info.filename in {
                           "code_manifest.json", "pyproject.toml", "README.md", "LICENSE",
                           "notebooks/02_oasis2_clinical_validation.ipynb",
                           "notebooks/03_oasis2_full_quality_audit.ipynb",
                           "docs/oasis2_full_quality_audit.md",
                       })
            if not allowed:
                raise ValueError("non_code_archive_member")
        manifest = json.loads(archive.read("code_manifest.json"))
        if (manifest.get("schema_version") != 1 or manifest.get("code_only") is not True
                or set(names) != set(manifest["files"]) | {"code_manifest.json"}):
            raise ValueError("invalid_code_manifest")
        blobs = {name: archive.read(name) for name in names}
        for name, digest in manifest["files"].items():
            if hashlib.sha256(blobs[name]).hexdigest() != digest:
                raise ValueError("code_checksum_mismatch")
    if destination.is_symlink():
        raise ValueError("unsafe_code_destination")
    if destination.exists():
        for name, content in blobs.items():
            path = destination / name
            if path.is_symlink() or path.read_bytes() != content:
                raise ValueError("existing_code_snapshot_differs_restart_runtime")
    else:
        destination.mkdir(parents=True)
        for name, content in blobs.items():
            path = destination / name
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(content)  # Public source code only; not a data/report writer.
    return manifest

In [ ]:
import subprocess

try:
    if sys.version_info < (3, 12):
        raise RuntimeError("Use Python 3.12+.")
    if INSTALL_PUBLIC_DEPENDENCIES:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *PUBLIC_PACKAGES],
            check=True, capture_output=True,
        )
    if not CODE_ROOT.exists():
        from google.colab import files

        uploaded = files.upload()  # Upload ONLY the reviewed public code archive.
        if set(uploaded) != {BUNDLE_NAME}:
            raise ValueError("expected_one_reviewed_code_bundle")
        bundle_manifest = unpack_code_bundle(uploaded[BUNDLE_NAME], CODE_ROOT)
        del uploaded
    else:
        bundle_manifest = json.loads((CODE_ROOT / "code_manifest.json").read_bytes())
        for name, digest in bundle_manifest["files"].items():
            target = CODE_ROOT / name
            if (target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
                    or hashlib.sha256(target.read_bytes()).hexdigest() != digest):
                raise ValueError("code_snapshot_changed_restart_runtime")
    source = str(CODE_ROOT / "src")
    if source not in sys.path:
        sys.path.insert(0, source)
except Exception:
    raise RuntimeError("Code setup failed; check the reviewed archive/runtime privately.") from None

## 2. Private input and output configuration

Set the placeholders only in your private Colab copy. `MANIFEST` is the existing
JSON array export; no MMSE or stored linkage field is invented. Raw MRI paths
remain relative to `IMAGE_ROOT`. `OUTPUT_DIR` must be a dedicated new directory
or this tool's existing run directory, separate from raw images and clinical
metadata. It cannot contain unrelated files, symlinks or hardlinked artifacts.
Do not put this directory in a Git checkout or share/commit its contents.

`MAX_NEW_PAIRS=3` is a cautious first batch. Set it to `None` for the remaining
cohort, or e.g. `25` for repeated bounded batches. It caps **new checks per call**,
not the deterministic cohort plan. Dry-run reads metadata/paths only and writes
nothing. Set the access acknowledgment only after verifying source authorization.

In [ ]:
AUTHORIZED_PRIVATE_ACCESS = False
PRIVATE_ROOT = Path("/content/drive/MyDrive/PRIVATE")
MANIFEST = PRIVATE_ROOT / "verified_clinical/oasis2_clinical_manifest.json"
IMAGE_ROOT = PRIVATE_ROOT / "OASIS-2/raw"
OUTPUT_DIR = PRIVATE_ROOT / "quality_audits/oasis2_run_001"
FIELD_MAP = None  # Default exact OASIS-2 array export profile.
RECORDS_KEY = "records"  # Only for an explicitly configured object envelope.
DRY_RUN = True
MAX_NEW_PAIRS = 3  # None = all remaining pairs; 25 = at most 25 new checks per call.
RECOVER_STALE_LOCK = False  # Only after confirming the previous writer/runtime has stopped.

In [ ]:
if not AUTHORIZED_PRIVATE_ACCESS:
    raise RuntimeError("Set the access acknowledgment only in an authorized private Colab copy.")
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from importlib.metadata import version

import nibabel as nib
import numpy as np

from brainagenticthon.preprocessing.oasis2_full_quality_audit import AuditError, audit_cohort

if np.__version__ != version("numpy") or nib.__version__ != version("nibabel"):
    raise RuntimeError("Restart the runtime after installing the pinned packages.")
if tuple(f"{name}=={version(name)}" for name in ("numpy", "nibabel")) != PUBLIC_PACKAGES:
    raise RuntimeError("Use the reviewed package pins or review the change explicitly.")


def show_progress(report):
    if report["completed_pairs"] % 25 == 0:
        print(json.dumps({
            "completed_pairs": report["completed_pairs"],
            "remaining_pairs": report["remaining_pairs"],
            "computational_fail_pairs": report["results"]["computational_fail_pairs"],
        }))

## 3. Run or resume

Run dry-run first and compare metadata counts with the supplied 150 subjects,
373 visit keys and 1,368 pair records. Differences require review; no reference
count replaces input data. Then set `DRY_RUN=False`, choose the batch size and
rerun configuration and the cell below. Reusing the same output directory resumes
its committed prefix. No clinical evidence or split/label writer is invoked.

Each pair is validated through the shared validator, with existing resource,
path, spatial geometry, finite/nonconstant voxel and before/after byte checks.
Aggregate state is checkpointed immediately after each completed pair. On restart,
completed bytes are **stream-rehashed**, not revalidated or loaded as MRI arrays.
This adds Drive I/O but detects changes even if size/timestamps were preserved.
Changed manifest, bytes, code, Python/package versions or validation limits require
a **new output directory**; the old run is never reset silently.

Stop through Colab's interrupt control. The current uncommitted pair may repeat;
previously committed checks are skipped. After a hard runtime kill a stale lock
can remain. Confirm the previous runtime is stopped before temporarily setting
`RECOVER_STALE_LOCK=True`; reset it to `False` afterward. Never run two writers.

In [ ]:
try:
    report = audit_cohort(
        MANIFEST, IMAGE_ROOT, OUTPUT_DIR, dry_run=DRY_RUN, max_pairs=MAX_NEW_PAIRS,
        field_map=FIELD_MAP, records_key=RECORDS_KEY,
        recover_stale_lock=RECOVER_STALE_LOCK, progress=show_progress,
    )
except KeyboardInterrupt:
    print(json.dumps({"status": "interrupted", "resume_from_last_committed_checkpoint": True}))
except AuditError as error:
    print(json.dumps({"status": "audit_stopped", "code": str(error)}))
else:
    print(json.dumps(report, indent=2, sort_keys=True, allow_nan=False))
    print(json.dumps({"supplied_reference_counts_match": {
        "subjects": report["inventory"]["subjects"] == 150,
        "visits": report["inventory"]["clinical_visit_keys"] == 373,
        "image_pair_records": report["inventory"]["image_pair_records"] == 1368,
    }}))

## 4. Interpretation, artifacts and limitations

`checkpoint.json` is authoritative; `aggregate_report.json` is regenerated from it
if interrupted between writes. Both contain aggregates, a cursor and hashed job/
byte-prefix bindings, never source IDs, filenames, paths, clinical rows or images.
They remain private, including rare geometry counts and dataset fingerprints.
Atomic replacement/checksums limit incomplete writes; Google Drive's remote sync,
FUSE locking, mount stability and durability are not guaranteed by Python. Keep
one writer and stable originals. Do not manually edit checkpoints or delete locks
from an active run. If a checkpoint is corrupt, preserve it and use a new run.

A complete audit can contain failed pairs; inspect the aggregate failure categories
and do not equate completion with a pass. `all_cohort_computational_checks_passed`
requires full coverage, no blocked manifest entries, no computational failures
and verified byte preservation. Visual QC, clinical diagnoses and final Task A/C
labels always remain unapproved. Analyze geometry, native T1 provenance and motion/
brain coverage require expert review. `.mat` sidecars and non-singleton 4D series
remain unsupported. Preservation hashes cannot authenticate the original source
or detect plausible pre-existing corruption without trusted source checksums.

No patient-level findings, frozen splits or clinical labels are exported. Failures
need a separate authorized private investigation. This workflow does not establish
cross-source generalization, independent calibration or unseen-disease detection,
and does not change Task C's model, preprocessing, calibration or JSON contract.